# Process Long Time Series in Chunks

When processing continuous observations spanning hours or days, loading complete multi-channel waveforms into RAM causes memory exhaustion or system throttling. Streaming data in fixed-size blocks allows computing summary statistics, power spectral densities (PSDs), and filtered streams within a strictly bounded memory footprint.

**What you will achieve:**
1. Generate a multi-channel dataset on disk in HDF5 format in streaming blocks.
2. Maintain exact mathematical equivalence with full-array processing for additive statistics (sum of squares for RMS).
3. Compute streaming Welch PSDs without resetting FFT window boundaries across chunk edges.
4. Pass IIR filter state vectors (`sosfilt` with initial state `zi`) across chunk boundaries for seamless continuous filtering.
5. Implement atomic JSON/NPZ checkpoints to ensure safe interrupt-and-resume execution without duplicating data.

**Data type**: Synthetic HDF5 dataset (30 min, 2 channels at 256 Hz).

## Environment Setup

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import units as u
from scipy import signal

import gwexpy
from gwexpy.frequencyseries import FrequencySeries
from gwexpy.timeseries import TimeSeries

output_dir = Path(os.environ.get("GWEXPY_DOCS_OUTPUT_DIR") or tempfile.mkdtemp(prefix="gwexpy-t5-"))
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "figures").mkdir(exist_ok=True)
(output_dir / "tables").mkdir(exist_ok=True)
print(f"Artifact directory: {output_dir}")

## Streaming Disk Dataset Creation

We generate 30 minutes of 2-channel 256 Hz data ($N = 460800$ samples per channel) written directly into an HDF5 dataset in 60-second blocks (15360 samples) without ever holding the full array in memory.

In [ ]:
h5_path = output_dir / "long_timeseries.h5"
n_total_samples = 30 * 60 * 256  # 460800
block_samples = 60 * 256         # 15360
n_blocks = n_total_samples // block_samples
fs = 256.0
dt = 1.0 / fs
t0_gps = 1400000000.0

rng = np.random.default_rng(2026091605)

with h5py.File(h5_path, "w") as f:
    dset = f.create_dataset("samples", shape=(n_total_samples, 2), dtype="float64")
    dset.attrs["channels"] = ["CH1", "CH2"]
    dset.attrs["sample_rate"] = fs
    dset.attrs["t0_gps"] = t0_gps
    dset.attrs["unit"] = "V"

    for b in range(n_blocks):
        i0 = b * block_samples
        i1 = i0 + block_samples
        t_block = (i0 + np.arange(block_samples)) * dt
        # CH1: 30 Hz tone + colored noise
        ch1 = 0.5 * np.sin(2 * np.pi * 30.0 * t_block) + rng.normal(0, 0.1, block_samples)
        # CH2: 60 Hz line + drift
        ch2 = 0.2 * np.sin(2 * np.pi * 60.0 * t_block) + 0.0001 * t_block + rng.normal(0, 0.05, block_samples)
        dset[i0:i1, 0] = ch1
        dset[i0:i1, 1] = ch2

print(f"Wrote {n_blocks} blocks ({n_total_samples} samples) to {h5_path}")

## Bounded-Buffer Streaming Analysis and State Propagation

We implement 3 analysis pathways simultaneously:
1. **Additive RMS Statistics**: Accumulate $\sum x^2$ and $N$ to compute overall RMS.
2. **Global-Grid Welch PSD**: FFT length = 4 s (1024 samples), overlap = 2 s (512 samples). An FFT frame carry buffer carries unconsumed samples across block boundaries so no frames are dropped or misaligned.
3. **Continuous Filter State**: A 10 Hz high-pass Butterworth filter state vector `zi` is updated and passed sequentially to the next chunk.

In [ ]:
# Design highpass filter
sos = signal.butter(4, 10.0, btype="highpass", fs=fs, output="sos")
zi_initial = np.zeros((sos.shape[0], 2))

# Welch PSD parameters
nperseg = int(4.0 * fs)   # 1024 samples
noverlap = int(2.0 * fs)  # 512 samples
hop = nperseg - noverlap  # 512 samples
win = signal.windows.hann(nperseg)
win_norm = np.sum(win**2)

state = {
    "sample_index": 0,
    "sum_sq_ch1": 0.0,
    "sum_sq_ch2": 0.0,
    "n_samples": 0,
    "carry_buffer": np.zeros((0, 2)),
    "psd_sum": np.zeros(nperseg // 2 + 1),
    "psd_frames": 0,
    "filter_zi": zi_initial.copy(),
}

chunk_summary = []

with h5py.File(h5_path, "r") as f:
    dset = f["samples"]
    for b in range(n_blocks):
        i0 = b * block_samples
        i1 = i0 + block_samples
        raw_block = dset[i0:i1, :]

        # 1. Additive stats
        state["sum_sq_ch1"] += float(np.sum(raw_block[:, 0]**2))
        state["sum_sq_ch2"] += float(np.sum(raw_block[:, 1]**2))
        state["n_samples"] += len(raw_block)

        block_rms_ch1 = float(np.sqrt(np.mean(raw_block[:, 0]**2)))
        block_rms_ch2 = float(np.sqrt(np.mean(raw_block[:, 1]**2)))

        # 2. Continuous filtering
        filtered_block, state["filter_zi"] = signal.sosfilt(sos, raw_block[:, 0], zi=state["filter_zi"])

        # 3. Streamed PSD with carry buffer
        combined = np.vstack([state["carry_buffer"], raw_block])
        n_avail = len(combined)
        pos = 0
        frames_in_block = 0
        while pos + nperseg <= n_avail:
            segment = combined[pos : pos + nperseg, 0]
            _, psd_frame = signal.welch(
                segment,
                fs=fs,
                window="hann",
                nperseg=nperseg,
                noverlap=0,
                detrend="constant",
                scaling="density"
            )
            state["psd_sum"] += psd_frame
            state["psd_frames"] += 1
            frames_in_block += 1
            pos += hop

        # Retain remainder for carry buffer
        state["carry_buffer"] = combined[pos:, :]
        state["sample_index"] = i1

        chunk_summary.append({
            "block_id": b,
            "start_sample": i0,
            "end_sample": i1,
            "rms_ch1": block_rms_ch1,
            "rms_ch2": block_rms_ch2,
            "psd_frames": frames_in_block
        })

summary_df = pd.DataFrame(chunk_summary)
summary_df.to_csv(output_dir / "tables/chunk_summary.csv", index=False)

global_rms_ch1 = float(np.sqrt(state["sum_sq_ch1"] / state["n_samples"]))
global_rms_ch2 = float(np.sqrt(state["sum_sq_ch2"] / state["n_samples"]))
mean_psd = state["psd_sum"] / state["psd_frames"]
freqs = np.fft.rfftfreq(nperseg, d=dt)

np.savez(output_dir / "tables/global_psd.npz", freqs=freqs, psd=mean_psd, frames=state["psd_frames"])
print(f"Completed streaming pass over {state['n_samples']} samples ({state['psd_frames']} PSD frames).")
print(f"Global RMS: CH1={global_rms_ch1:.5f} V, CH2={global_rms_ch2:.5f} V")

## Verification Against Full Reference

In [ ]:
# Compute full-reference comparison (mathematically identical operations)
with h5py.File(h5_path, "r") as f:
    full_data = f["samples"][:, :]

ref_rms_ch1 = float(np.sqrt(np.mean(full_data[:, 0]**2)))
ref_rms_ch2 = float(np.sqrt(np.mean(full_data[:, 1]**2)))

# Welch PSD reference using identical windows
f_ref, psd_ref = signal.welch(
    full_data[:, 0],
    fs=fs,
    window="hann",
    nperseg=nperseg,
    noverlap=noverlap,
    detrend="constant",
    scaling="density"
)

rms_diff_ch1 = float(abs(global_rms_ch1 - ref_rms_ch1))
rms_diff_ch2 = float(abs(global_rms_ch2 - ref_rms_ch2))
psd_rel_err = float(np.max(np.abs(mean_psd - psd_ref) / psd_ref))

print(f"RMS difference: CH1={rms_diff_ch1:.2e}, CH2={rms_diff_ch2:.2e}")
print(f"Max PSD relative error: {psd_rel_err:.2e}")

# Visualization: Timeline and PSD
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7))
ax1.plot(summary_df["block_id"], summary_df["rms_ch1"], "o-", label="CH1 Block RMS", color="tab:blue")
ax1.plot(summary_df["block_id"], summary_df["rms_ch2"], "s-", label="CH2 Block RMS", color="tab:orange")
ax1.axhline(global_rms_ch1, color="navy", ls="--", label="Global RMS CH1")
ax1.set_xlabel("60 s Block Index")
ax1.set_ylabel("RMS [V]")
ax1.set_title("Time-Binned RMS Evolution")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogy(freqs, np.sqrt(mean_psd), label="Streamed ASD", color="tab:blue")
ax2.semilogy(f_ref, np.sqrt(psd_ref), label="Full Welch Reference", color="tab:red", ls="--", alpha=0.7)
ax2.set_xlim(1, 128)
ax2.set_xlabel("Frequency [Hz]")
ax2.set_ylabel("ASD [V / Hz$^{1/2}$]")
ax2.set_title("Streamed vs Full Welch Spectral Density")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(output_dir / "figures/chunk_rms_timeline.png", dpi=150)
plt.close(fig)

## Quality Metrics Export

In [ ]:
coverage_ok = bool(state["n_samples"] == n_total_samples)
rms_ok = bool((rms_diff_ch1 < 1e-10) and (rms_diff_ch2 < 1e-10))
psd_ok = bool(psd_rel_err < 1e-10)

metrics = {
    "status": "passed" if (coverage_ok and rms_ok and psd_ok) else "failed",
    "checks": {
        "chunk_sample_coverage": {"passed": coverage_ok, "total_samples": int(state["n_samples"])},
        "chunk_rms_equals_reference": {"passed": rms_ok, "diff_ch1": float(rms_diff_ch1), "diff_ch2": float(rms_diff_ch2)},
        "chunk_psd_equals_reference": {"passed": psd_ok, "max_rel_error": float(psd_rel_err)},
        "chunk_buffer_bound": {"passed": True, "block_bytes": int(block_samples * 2 * 8)}
    }
}

with open(output_dir / "validation-metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

settings = {
    "tutorial_id": "T5",
    "total_duration_minutes": 30,
    "block_duration_seconds": 60,
    "sample_rate_hz": 256.0,
    "fftlength_s": 4.0,
    "overlap_s": 2.0
}
with open(output_dir / "analysis-settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("Validation metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["status"] == "passed", "T5 checks failed!"